# Advanced Applications in Math and Python

This notebook synthesizes statistics, linear algebra, probability, and
Python libraries (NumPy, Pandas, Matplotlib) to solve multi-step applied
problems — no machine learning involved.

**Contents**
1. Setup
2. Part 1.1 — Covariance & Correlation Matrices (manual vs. built-in)
3. Part 1.2 — Markov Chains: simulation & steady-state via eigenvectors
4. Part 2.1 — Time-Series Windowing (rolling mean, expanding std, NaN handling)
5. Part 2.2 — Advanced Visualization (conditional scatter + shaded rolling band)


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline


## 2. Part 1.1 — Covariance and Correlation Matrices

We simulate 3 variables (e.g. 3 sensor channels) with 100 observations each,
drawn from a normal distribution, then compute the covariance matrix two
ways:

1. **Manually**, using the matrix-multiplication definition
   $$\Sigma = \frac{1}{n-1} (X - \bar{X})(X - \bar{X})^T$$
2. **Via NumPy's built-in** `np.cov`, to verify the manual result.


In [ ]:
# 3 variables x 100 observations, each from a (slightly different) normal distribution
n_vars, n_obs = 3, 100
means = [0, 5, -2]
stds = [1.0, 2.0, 0.5]

X = np.array([
    np.random.normal(loc=means[i], scale=stds[i], size=n_obs)
    for i in range(n_vars)
])  # shape: (3, 100) -> rows = variables, columns = observations

print("X shape:", X.shape)
X[:, :5]  # peek at first 5 observations per variable


In [ ]:
# --- Manual covariance matrix via matrix multiplication ---
# 1. Center each variable (subtract its row mean)
X_mean = X.mean(axis=1, keepdims=True)          # shape (3, 1)
X_centered = X - X_mean                          # shape (3, 100)

# 2. Covariance = (X_centered @ X_centered.T) / (n_obs - 1)
cov_manual = (X_centered @ X_centered.T) / (n_obs - 1)

print("Manual covariance matrix:")
print(np.round(cov_manual, 4))


In [ ]:
# --- Verify against NumPy's built-in function ---
cov_builtin = np.cov(X)  # np.cov treats rows as variables by default

print("Built-in (np.cov) covariance matrix:")
print(np.round(cov_builtin, 4))

print("\nMax absolute difference between manual and built-in:",
      np.max(np.abs(cov_manual - cov_builtin)))

assert np.allclose(cov_manual, cov_builtin), "Manual and built-in covariance matrices should match!"
print("Manual calculation matches np.cov ✓")


In [ ]:
# --- Bonus: derive the correlation matrix from the covariance matrix ---
std_devs = np.sqrt(np.diag(cov_manual))
corr_manual = cov_manual / np.outer(std_devs, std_devs)

corr_builtin = np.corrcoef(X)

print("Manual correlation matrix:")
print(np.round(corr_manual, 4))
print("\nBuilt-in (np.corrcoef) correlation matrix:")
print(np.round(corr_builtin, 4))

assert np.allclose(corr_manual, corr_builtin), "Manual and built-in correlation matrices should match!"
print("\nManual correlation matches np.corrcoef ✓")


## 3. Part 1.2 — Markov Chains

A 3-state weather system: **Sunny, Cloudy, Rainy**. Rows of the transition
matrix `P` sum to 1 — `P[i, j]` is the probability of moving from state `i`
to state `j` on the next step.

We (a) simulate 50 discrete transitions starting from a chosen state, and
(b) find the long-run steady-state distribution analytically via the
eigenvectors of `P.T` (the eigenvector associated with eigenvalue 1, in
transposed form, is the stationary distribution).


In [ ]:
states = ["Sunny", "Cloudy", "Rainy"]

# Rows: current state -> Columns: next state. Each row sums to 1.
P = np.array([
    [0.7, 0.2, 0.1],   # from Sunny
    [0.3, 0.4, 0.3],   # from Cloudy
    [0.2, 0.3, 0.5],   # from Rainy
])

assert np.allclose(P.sum(axis=1), 1.0), "Each row of a transition matrix must sum to 1"
print("Transition matrix P:")
print(pd.DataFrame(P, index=states, columns=states))


In [ ]:
# --- (a) Simulate 50 steps starting from a specific state ---
def simulate_markov_chain(P, start_state_idx, n_steps=50, seed=0):
    rng = np.random.default_rng(seed)
    n_states = P.shape[0]
    state = start_state_idx
    trajectory = [state]

    for _ in range(n_steps):
        state = rng.choice(n_states, p=P[state])
        trajectory.append(state)

    return np.array(trajectory)


start_state = 0  # start in "Sunny"
trajectory = simulate_markov_chain(P, start_state, n_steps=50, seed=42)
trajectory_named = [states[i] for i in trajectory]

print("Simulated 50-step trajectory (starting from Sunny):")
print(trajectory_named)


In [ ]:
# Visualize the simulated trajectory
plt.figure(figsize=(10, 3))
plt.plot(trajectory, marker="o", linestyle="-", markersize=4)
plt.yticks(range(len(states)), states)
plt.xlabel("Step")
plt.ylabel("State")
plt.title("Simulated Markov Chain Trajectory (50 steps)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- (b) Steady-state distribution via eigenvectors of P.T ---
eigenvalues, eigenvectors = np.linalg.eig(P.T)

print("Eigenvalues of P.T:", np.round(eigenvalues, 4))

# The stationary distribution corresponds to the eigenvalue equal to 1
steady_idx = np.argmin(np.abs(eigenvalues - 1.0))
steady_state_vector = np.real(eigenvectors[:, steady_idx])

# Normalize so the probabilities sum to 1
steady_state = steady_state_vector / steady_state_vector.sum()

print("\nSteady-state probabilities:")
for s, p in zip(states, steady_state):
    print(f"  {s:>6}: {p:.4f}")

assert np.isclose(steady_state.sum(), 1.0)


In [ ]:
# --- Sanity check: steady state should be a fixed point, i.e. steady_state @ P == steady_state ---
check = steady_state @ P
print("steady_state @ P =", np.round(check, 4))
print("steady_state     =", np.round(steady_state, 4))
print("Match:", np.allclose(check, steady_state))

# Also cross-check empirically: the long-run fraction of time spent in each state
# during a much longer simulation should approach the analytical steady state.
long_run = simulate_markov_chain(P, start_state, n_steps=20000, seed=123)
empirical_freqs = np.bincount(long_run, minlength=3) / len(long_run)
print("\nEmpirical long-run frequencies (20,000 steps):", np.round(empirical_freqs, 4))
print("Analytical steady state:                        ", np.round(steady_state, 4))


## 4. Part 2.1 — Time-Series Windowing

We build a `DatetimeIndex`-based DataFrame spanning a full calendar year
(365 days) of simulated daily data, then compute:

- A **7-day rolling mean**
- A **30-day expanding standard deviation**

...and handle the `NaN`s that rolling/expanding windows introduce at the
start of the series before they have enough data to compute a value.


In [ ]:
# Simulate 365 days of daily data with a mild upward trend + noise + weekly seasonality
n_days = 365
date_index = pd.date_range(start="2025-01-01", periods=n_days, freq="D")

t = np.arange(n_days)
trend = 0.02 * t
seasonality = 3 * np.sin(2 * np.pi * t / 7)  # weekly pattern
noise = np.random.normal(loc=0, scale=2, size=n_days)

daily_values = 50 + trend + seasonality + noise

df = pd.DataFrame({"value": daily_values}, index=date_index)
df.index.name = "date"

print(df.shape)
df.head(10)


In [ ]:
# --- Rolling mean (7-day window) and expanding std (min 30 observations) ---
df["rolling_mean_7d"] = df["value"].rolling(window=7).mean()
df["expanding_std_30d"] = df["value"].expanding(min_periods=30).std()

print("NaNs introduced by rolling_mean_7d:", df["rolling_mean_7d"].isna().sum())
print("NaNs introduced by expanding_std_30d:", df["expanding_std_30d"].isna().sum())

df.head(10)


In [ ]:
# --- Systematically handle the resulting NaNs ---
# Strategy:
#   - rolling_mean_7d: back-fill the first 6 leading NaNs with the first valid
#     rolling value, since there's no earlier data to average over.
#   - expanding_std_30d: leave the leading NaNs as NaN (there genuinely isn't
#     enough data yet for a meaningful std estimate) but expose a boolean
#     mask/flag column so downstream code can filter or handle them explicitly
#     instead of silently propagating NaNs.

df["rolling_mean_7d_filled"] = df["rolling_mean_7d"].bfill()
df["expanding_std_30d_valid"] = df["expanding_std_30d"].notna()

# Alternative / additional strategy: drop rows where ANY windowed column is NaN,
# useful when you need a fully-populated matrix for downstream analysis.
df_complete = df.dropna(subset=["rolling_mean_7d", "expanding_std_30d"])

print(f"Original rows: {len(df)} | Rows with all windowed features valid: {len(df_complete)}")
df.head(10)


## 5. Part 2.2 — Advanced Visualization

A single figure, 2 subplots side-by-side:

- **Left:** Scatter plot of the raw daily values, with points more than
  **2 standard deviations** from the overall mean highlighted in a
  different color (the empirical/68-95-99.7 rule).
- **Right:** Line plot of the raw daily data overlaid with the 7-day
  rolling mean, with a shaded band (`fill_between`) showing
  mean ± 1 rolling-window standard deviation.


In [ ]:
overall_mean = df["value"].mean()
overall_std = df["value"].std()

lower_bound = overall_mean - 2 * overall_std
upper_bound = overall_mean + 2 * overall_std

is_outlier = (df["value"] < lower_bound) | (df["value"] > upper_bound)
print(f"Overall mean: {overall_mean:.2f} | std: {overall_std:.2f}")
print(f"2-sigma bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Points flagged as outliers (> 2 std from mean): {is_outlier.sum()} / {len(df)}")


In [ ]:
# For the shaded band on the right subplot: rolling mean +/- rolling std (7-day window)
rolling_std_7d = df["value"].rolling(window=7).std()

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# ---------------- Subplot 1: conditional-color scatter ----------------
ax1 = axes[0]
colors = np.where(is_outlier, "crimson", "steelblue")

ax1.scatter(df.index[~is_outlier], df["value"][~is_outlier],
            color="steelblue", s=18, alpha=0.7, label="Within ±2σ")
ax1.scatter(df.index[is_outlier], df["value"][is_outlier],
            color="crimson", s=30, alpha=0.9, label="Outside ±2σ", zorder=3)

ax1.axhline(overall_mean, color="black", linestyle="--", linewidth=1, label="Mean")
ax1.axhline(upper_bound, color="gray", linestyle=":", linewidth=1, label="Mean ± 2σ")
ax1.axhline(lower_bound, color="gray", linestyle=":", linewidth=1)

ax1.set_title("Daily Values with 2σ Outlier Highlighting")
ax1.set_xlabel("Date")
ax1.set_ylabel("Value")
ax1.legend(loc="upper left", fontsize=8)
ax1.tick_params(axis="x", rotation=30)

# ---------------- Subplot 2: raw data + rolling mean + shaded std band ----------------
ax2 = axes[1]

ax2.plot(df.index, df["value"], color="lightgray", linewidth=1, label="Raw daily value")
ax2.plot(df.index, df["rolling_mean_7d"], color="darkorange", linewidth=2, label="7-day rolling mean")

ax2.fill_between(
    df.index,
    df["rolling_mean_7d"] - rolling_std_7d,
    df["rolling_mean_7d"] + rolling_std_7d,
    color="darkorange", alpha=0.2, label="±1 rolling std"
)

ax2.set_title("Raw Data vs. 7-Day Rolling Mean (± 1 std band)")
ax2.set_xlabel("Date")
ax2.set_ylabel("Value")
ax2.legend(loc="upper left", fontsize=8)
ax2.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## 6. Summary

- **Part 1.1** confirmed the matrix-multiplication definition of covariance
  matches NumPy's built-in `np.cov` exactly, and extended it to correlation.
- **Part 1.2** simulated a 3-state Markov chain and showed the analytical
  steady state (via eigen-decomposition of `P.T`) agrees with the empirical
  long-run visitation frequencies from a long simulation.
- **Part 2.1** built a full-year `DatetimeIndex` series, computed rolling and
  expanding statistics, and applied explicit, documented strategies for the
  resulting leading `NaN`s rather than silently dropping or ignoring them.
- **Part 2.2** combined the empirical rule (±2σ) and rolling-window
  statistics into a single two-panel Matplotlib figure with conditional
  coloring, a shaded uncertainty band, and full legends/labels.
